# 02 模型与组合研究

按顺序运行：时间切分、单日模型、序列模型、共同样本评价、图表。
YEAR=2025 对应本次已运行的版本；改为2024可复现验证年流程。
已保存的两年结果与对比图见仓库 README，无需为了看结果重新训练。


## 1. 读取01保存的一张表

In [ ]:
from pathlib import Path
import gc
import math
import random
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression, Ridge
from lightgbm import LGBMRegressor
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader

torch.set_num_threads(2)
ROOT = Path('data/ml_alpha_simple')
YEAR = 2025
LOOKBACK = 20
EPOCHS = 3
SEEDS = [42, 7, 2026]
TARGET_SCALE = 100.0
FEATURES = ['momentum_2', 'momentum_5', 'volatility_5', 'volume_ratio_5', 'intraday_return']
RESULT = ROOT / ('results_' + str(YEAR))
RESULT.mkdir(parents=True, exist_ok=True)

data = pd.read_csv(str(ROOT / 'model_data.csv.gz'),
                   parse_dates=['date', 'label_end_date'], dtype={'stock': str})
data = data.sort_values(['stock', 'date']).reset_index(drop=True)
# 兼容CSV将布尔值识别为字符串的情况。
data['signal_eligible'] = data['signal_eligible'].map(
    {True: True, False: False, 'True': True, 'False': False})
assert data['signal_eligible'].notna().all()
data['signal_eligible'] = data['signal_eligible'].astype(bool)
assert not data.duplicated(['date', 'stock']).any()
print('建模表：', data.shape)


## 2. 时间切分

训练从2020开始，截止评价年的前一年。信号日期和标签结束日期都必须在各自区间内。
这里是扩展窗口：YEAR=2022、2023、2024、2025分别对应不同训练截止年。

In [ ]:
train_end = pd.Timestamp('{}-12-31'.format(YEAR - 1))
eval_start = pd.Timestamp('{}-01-01'.format(YEAR))
eval_end = pd.Timestamp('{}-12-31'.format(YEAR))
finite_label = np.isfinite(data['label'])
train_mask = (data['signal_eligible'] & finite_label
              & data['date'].between(pd.Timestamp('2020-01-01'), train_end)
              & data['label_end_date'].le(train_end))
eval_mask = (data['signal_eligible'] & finite_label
             & data['date'].between(eval_start, eval_end)
             & data['label_end_date'].le(eval_end))
assert train_mask.any() and eval_mask.any()
for name, mask in [('训练', train_mask), ('评价', eval_mask)]:
    subset = data.loc[mask, ['date', 'label_end_date']]
    print(name, '行数：', len(subset), '信号：', subset['date'].min(), subset['date'].max(),
          '标签最晚结束：', subset['label_end_date'].max())


## 3. 合法20日窗口

完整窗口必须包含同一股票的20个连续交易日，且所有输入特征有限。
先统一评价样本，所有模型和基准都评价这些行。单日模型仍使用全部合法训练行。

In [ ]:
def require(condition, message):
    if not bool(condition):
        raise ValueError(message)

def sequence_window_mask(frame, lookback):
    require(not frame.duplicated(['stock', 'date']).any(), '序列表组合键重复。')
    finite = np.isfinite(frame[FEATURES].values).all(axis=1).astype(np.int64)
    cumulative = np.concatenate(([0], np.cumsum(finite)))
    ends = np.arange(len(frame))
    starts = np.maximum(ends - lookback + 1, 0)
    complete = cumulative[ends + 1] - cumulative[starts] == lookback
    codes = pd.factorize(frame['stock'], sort=False)[0]
    same_stock = codes[ends] == codes[starts]
    calendar = pd.DatetimeIndex(frame['date'].unique()).sort_values()
    positions = calendar.get_indexer(frame['date'])
    continuous = positions[ends] - positions[starts] == lookback - 1
    return complete & same_stock & continuous & (ends >= lookback - 1)


In [ ]:
window_ok = sequence_window_mask(data, LOOKBACK)
train_ends = np.flatnonzero(train_mask.values & window_ok)
eval_ends = np.flatnonzero(eval_mask.values & window_ok)
assert len(train_ends) > 0 and len(eval_ends) > 0
predictions = data.iloc[eval_ends][['date', 'stock', 'label', 'label_end_date', 'momentum_2']].copy()
predictions = predictions.reset_index(drop=True)
print('序列训练：', len(train_ends), '共同评价：', len(eval_ends))
print('因窗口不完整排除的评价行：', int(eval_mask.sum()) - len(eval_ends))


## 4. 单日特征标准化

只用训练集估计均值和标准差；评价集使用同一组参数。

In [ ]:
X_train = data.loc[train_mask, FEATURES]
y_train = data.loc[train_mask, 'label']
train_mean = X_train.mean()
train_std = X_train.std(ddof=0)
assert np.isfinite(train_mean).all() and np.isfinite(train_std).all() and train_std.gt(0).all()
X_train_scaled = (X_train - train_mean) / train_std
X_eval_scaled = (data.iloc[eval_ends][FEATURES] - train_mean) / train_std


## 5. OLS、Ridge、LightGBM

每个模型都重新拟合。LightGBM参数沿用原实验，不依据本次评价结果调参。

In [ ]:
models = {
    'ols': LinearRegression(),
    'ridge': Ridge(alpha=1.0),
    'lightgbm': LGBMRegressor(**{'learning_rate': 0.05, 'max_depth': 4, 'min_child_samples': 200, 'n_estimators': 200, 'n_jobs': 2, 'num_leaves': 15, 'objective': 'regression', 'random_state': 42})
}
for name, model in models.items():
    model.fit(X_train_scaled, y_train)
    predictions[name] = model.predict(X_eval_scaled)
    assert np.isfinite(predictions[name]).all()
    print(name, '完成', flush=True)
del X_train, y_train, X_train_scaled, X_eval_scaled
gc.collect()


## 6. 序列标准化与Dataset

序列模型的均值/标准差由合法训练序列的末端行估计，沿用原实验口径。
Dataset每次按末端位置取20×5窗口，标签取末端行；训练标签乘100，输出之后除100。

In [ ]:
sequence_train = data.iloc[train_ends][FEATURES]
sequence_mean = sequence_train.mean().values.astype(np.float32)
sequence_std = sequence_train.std(ddof=0).values.astype(np.float32)
assert np.isfinite(sequence_mean).all() and np.isfinite(sequence_std).all() and (sequence_std > 0).all()
features = data[FEATURES].values.astype(np.float32)
features = (features - sequence_mean) / sequence_std
labels = data['label'].values.astype(np.float32)
del sequence_train


In [ ]:
class StockSequenceDataset(Dataset):
    def __init__(self, features, labels, end_indices, lookback, target_scale):
        self.features = features
        self.labels = labels
        self.end_indices = np.asarray(end_indices, dtype=np.int64)
        self.lookback = lookback
        self.target_scale = target_scale

    def __len__(self):
        return len(self.end_indices)

    def __getitem__(self, item):
        end = int(self.end_indices[item])
        x = self.features[end - self.lookback + 1:end + 1]
        y = self.labels[end] * self.target_scale
        return torch.from_numpy(x), torch.tensor(y, dtype=torch.float32)


In [ ]:
train_dataset = StockSequenceDataset(features, labels, train_ends, LOOKBACK, TARGET_SCALE)
eval_dataset = StockSequenceDataset(features, labels, eval_ends, LOOKBACK, TARGET_SCALE)
x, y = train_dataset[0]
print('单条输入：', tuple(x.shape), '训练标签：', y.item())


## 7. 同窗口的线性参照

窗口Ridge也使用20×5历史，避免把信息量差异误认为模型结构的优势。

In [ ]:
offsets = np.arange(-LOOKBACK + 1, 1)

X_window_train = features[
    train_ends[:, None] + offsets
].reshape(len(train_ends), -1)

y_window_train = (
    labels[train_ends] * np.float32(TARGET_SCALE)
)

window_model = Ridge(alpha=1.0, fit_intercept=True)
window_model.fit(X_window_train, y_window_train)

del X_window_train, y_window_train
gc.collect()

X_window_eval = features[
    eval_ends[:, None] + offsets
].reshape(len(eval_ends), -1)

predictions['window_ridge'] = (
    window_model.predict(X_window_eval) / TARGET_SCALE
)

del X_window_eval
gc.collect()

print('窗口Ridge完成')


## 8. LSTM与Transformer结构

两种网络都把20×5的历史窗口映射为一个未来收益预测。
LSTM使用最后时刻的隐藏表示；Transformer先投影到16维、加入位置表示，
再经过双头注意力和前馈层，取最后一个位置输出。每层残差连接后做LayerNorm。
窗口截至信号日收盘，所有位置都是已知历史；仅预测窗口末端的标签，因此这里不加因果掩码。


In [ ]:
class AlphaLSTM(nn.Module):
    def __init__(self, input_size=5, hidden_size=16):
        super(AlphaLSTM, self).__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers=1, batch_first=True)
        self.head = nn.Linear(hidden_size, 1)

    def forward(self, x):
        output, _ = self.lstm(x)
        return self.head(output[:, -1, :]).squeeze(-1)


In [ ]:
class AlphaTransformer(nn.Module):
    def __init__(self, input_size=5, lookback=20, d_model=16, n_heads=2, ff_size=32):
        super(AlphaTransformer, self).__init__()
        require(d_model % n_heads == 0, '注意力维度不能整除头数。')
        self.lookback, self.d_model = lookback, d_model
        self.n_heads, self.head_dim = n_heads, d_model // n_heads
        self.input_projection = nn.Linear(input_size, d_model)
        self.position = nn.Parameter(torch.randn(1, lookback, d_model) * 0.02)
        self.q_projection = nn.Linear(d_model, d_model)
        self.k_projection = nn.Linear(d_model, d_model)
        self.v_projection = nn.Linear(d_model, d_model)
        self.attention_projection = nn.Linear(d_model, d_model)
        self.norm1 = nn.LayerNorm(d_model)
        self.feed_forward = nn.Sequential(nn.Linear(d_model, ff_size), nn.ReLU(),
                                          nn.Linear(ff_size, d_model))
        self.norm2 = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, 1)

    def forward(self, x):
        batch, length, _ = x.size()
        require(length == self.lookback, '输入窗口长度不一致。')
        z = self.input_projection(x) + self.position
        q = self.q_projection(z).view(batch, length, self.n_heads, self.head_dim).transpose(1, 2)
        k = self.k_projection(z).view(batch, length, self.n_heads, self.head_dim).transpose(1, 2)
        v = self.v_projection(z).view(batch, length, self.n_heads, self.head_dim).transpose(1, 2)
        weights = torch.softmax(torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(self.head_dim), dim=-1)
        context = torch.matmul(weights, v).transpose(1, 2).contiguous().view(batch, length, self.d_model)
        z = self.norm1(z + self.attention_projection(context))
        z = self.norm2(z + self.feed_forward(z))
        return self.head(z[:, -1, :]).squeeze(-1)


## 9. 固定3轮训练

重新初始化每个模型。DataLoader只打乱已划入训练集的完整样本窗口，不打乱窗口内部日期。
不使用评价年选轮数、不恢复检查点。训练函数只做前向、损失、反向、参数更新。

In [ ]:
def train_network(network, dataset, name):
    loader = DataLoader(dataset, batch_size=128, shuffle=True, num_workers=0)
    optimizer = torch.optim.Adam(network.parameters(), lr=0.001)
    criterion = nn.MSELoss()
    logs = []
    for epoch in range(1, EPOCHS + 1):
        network.train()
        loss_sum, count = 0.0, 0
        started = time.time()
        for x, y in loader:
            optimizer.zero_grad()
            pred = network(x)
            assert pred.shape == y.shape
            loss = criterion(pred, y)
            assert torch.isfinite(loss).item(), '训练损失无效'
            loss.backward()
            optimizer.step()
            loss_sum += loss.item() * x.size(0)
            count += x.size(0)
        logs.append({'model': name, 'epoch': epoch, 'loss': loss_sum / count,
                     'seconds': time.time() - started})
        print(name, 'epoch', epoch, 'loss', loss_sum / count, flush=True)
    return logs


def predict_network(network, dataset):
    loader = DataLoader(dataset, batch_size=256, shuffle=False, num_workers=0)
    network.eval()
    parts = []
    with torch.no_grad():
        for x, _ in loader:
            parts.append(network(x).cpu().numpy())
    values = np.concatenate(parts) / TARGET_SCALE
    assert np.isfinite(values).all()
    return values


In [ ]:
training_logs = []
for family, constructor in [('lstm', AlphaLSTM), ('transformer', AlphaTransformer)]:
    for seed in SEEDS:
        random.seed(seed)
        np.random.seed(seed)
        torch.manual_seed(seed)
        name = '{}_seed{}'.format(family, seed)
        network = constructor()
        training_logs.extend(train_network(network, train_dataset, name))
        predictions[name] = predict_network(network, eval_dataset)
        del network
        gc.collect()
training_logs = pd.DataFrame(training_logs)
display(training_logs)


## 10. 逐日Rank IC

模型必须在相同的日期与股票样本上比较。三种种子全部报告。

In [ ]:
def summarize_predictions(predictions, names):
    daily_ics = {}
    for name in names:

        daily_ics[name] = pd.Series({
            date: group[name].rank(method='average').corr(group['label'].rank(method='average'))
            for date, group in predictions.groupby('date')})
    table = pd.DataFrame(daily_ics).sort_index()
    table.index.name = 'date'
    summary = pd.DataFrame({'valid_days': table.count(), 'mean_rank_ic': table.mean(),
                            'std_rank_ic': table.std(ddof=1),
                            'positive_ratio': table.gt(0).sum() / table.count()})
    return table, summary


In [ ]:
names = ['momentum_2', 'ols', 'ridge', 'lightgbm', 'window_ridge'] + [
    '{}_seed{}'.format(family, seed) for family in ['lstm', 'transformer'] for seed in SEEDS
]
daily_ic, ic_summary = summarize_predictions(predictions, names)
quarterly_ic = daily_ic.groupby(daily_ic.index.to_period('Q')).mean()
display(ic_summary)
display(quarterly_ic)


## 11. 从预测到持仓、成本和净值

高分五分位等权。下一次交易前权重先按已实现收益漂移；首次建仓计成本，期末不额外清仓。
买卖各5基点，一阶近似扣费。基准是同一可评价股票池等权，不是官方指数。
未来停牌不提前筛掉，因此这是理想成交下的研究组合，不是完整实盘回测。

In [ ]:
def evaluate_portfolio(predictions, score, cost_bps=5, annualization=252):
    data = predictions.sort_values(['date', 'stock']).copy()
    require(np.isfinite(data['label']).all(), '组合样本标签无效。')
    require(not data.duplicated(['date', 'stock']).any(), '组合样本重复。')
    groups = None
    if score is not None:
        require(np.isfinite(data[score]).all(), '组合预测分数无效。')
        require(data.groupby('date')['stock'].count().ge(5).all(), '当日股票数不足5只。')
        data['group'] = data.groupby('date')[score].transform(
            lambda s: pd.qcut(s.rank(method='first'), 5, labels=False))
        groups = data.groupby(['date', 'group'])['label'].mean().unstack('group')
        groups.columns = ['G{}'.format(int(value) + 1) for value in groups.columns]
        selected = data.loc[data['group'].eq(4)].copy()
    else:
        selected = data.copy()
    selected['weight'] = 1.0 / selected.groupby('date')['stock'].transform('count')
    weights = selected.pivot(index='date', columns='stock', values='weight').fillna(0).sort_index()
    returns = data.pivot(index='date', columns='stock', values='label').reindex(
        index=weights.index, columns=weights.columns)
    require(not ((weights > 0) & returns.isna()).any().any(), '持仓股票缺少收益率。')
    returns = returns.fillna(0.0)  # 此处缺失格子的目标权重为0，不用于填补真实持仓收益。
    require(np.allclose(weights.sum(axis=1), 1.0), '目标权重之和不为1。')
    gross = (weights * returns).sum(axis=1)
    require(gross.gt(-1).all(), '组合收益不大于-100%，无法按此方法计算权重漂移。')
    end_weights = (weights * (1.0 + returns)).div(1.0 + gross, axis=0)
    trades = weights - end_weights.shift(1).fillna(0.0)
    daily = pd.DataFrame({'gross_return': gross,
                          'buy_ratio': trades.clip(lower=0).sum(axis=1),
                          'sell_ratio': -trades.clip(upper=0).sum(axis=1)})
    daily['traded_ratio'] = daily['buy_ratio'] + daily['sell_ratio']
    daily['cost_ratio'] = daily['traded_ratio'] * cost_bps / 10000.0
    daily['net_return_approx'] = daily['gross_return'] - daily['cost_ratio']
    require(daily['net_return_approx'].gt(-1).all(), '近似扣费导致收益不大于-100%。')
    nav = (1.0 + daily[['gross_return', 'net_return_approx']]).cumprod()
    drawdown = nav / nav.cummax().clip(lower=1.0) - 1.0
    net_std = daily['net_return_approx'].std(ddof=1)
    n = len(daily)
    summary = pd.Series({
        'periods': n,
        'gross_total_return': nav['gross_return'].iloc[-1] - 1,
        'net_total_return_approx': nav['net_return_approx'].iloc[-1] - 1,
        'gross_max_drawdown': drawdown['gross_return'].min(),
        'net_max_drawdown': drawdown['net_return_approx'].min(),
        'mean_traded_ratio_ex_initial': daily['traded_ratio'].iloc[1:].mean(),
        'mean_period_cost': daily['cost_ratio'].mean(),
        'net_annualized_return_approx': nav['net_return_approx'].iloc[-1] ** (annualization / float(n)) - 1,
        'net_annualized_volatility': net_std * math.sqrt(annualization),
        'net_sharpe_zero_rf': (daily['net_return_approx'].mean() / net_std * math.sqrt(annualization)
                               if net_std > 0 else np.nan)
    })
    return {'daily': daily, 'nav': nav, 'drawdown': drawdown,
            'summary': summary, 'groups': groups,
            'positions': selected[['date', 'stock', 'weight']]}


In [ ]:
results = {}
for name in names + ['universe_equal_weight']:
    score = None if name == 'universe_equal_weight' else name
    results[name] = evaluate_portfolio(predictions, score, cost_bps=5)
summary = pd.DataFrame({name: result['summary'] for name, result in results.items()}).T
summary = summary.join(ic_summary[['mean_rank_ic', 'std_rank_ic', 'positive_ratio']])
summary['net_return_difference_vs_benchmark'] = (summary['net_total_return_approx']
    - summary.loc['universe_equal_weight', 'net_total_return_approx'])
display(summary)
# 标签结束日期作为收益实现日期。
date_map = predictions[['date', 'label_end_date']].drop_duplicates().set_index('date')['label_end_date']
assert date_map.index.is_unique
net_nav = pd.DataFrame({name: result['nav']['net_return_approx'] for name, result in results.items()})
net_nav.index = pd.DatetimeIndex(date_map.loc[net_nav.index])
net_nav.index.name = 'pnl_end_date'
initial = pd.DataFrame(1.0, index=[net_nav.index[0] - pd.Timedelta(days=1)], columns=net_nav.columns)
plot_nav = pd.concat([initial, net_nav])


## 12. 图表

主图显示所有种子，不按评价结果挑选。

In [ ]:
plt.rcParams.update({'figure.dpi': 110, 'font.size': 9})
base_names = ['momentum_2', 'ols', 'ridge', 'lightgbm', 'window_ridge', 'universe_equal_weight']
sequence_names = [name for name in names if name.startswith(('lstm_', 'transformer_'))] + ['universe_equal_weight']
fig, axes = plt.subplots(2, 1, figsize=(12, 9), sharex=True)
for ax, selected in zip(axes, [base_names, sequence_names]):
    plot_nav[selected].plot(ax=ax)
    ax.set_ylabel('Net NAV')
    ax.grid(alpha=0.2)
fig.tight_layout()
fig.savefig(str(RESULT / 'net_nav.png'))
plt.show()

fig, ax = plt.subplots(figsize=(12, 5))
(plot_nav / plot_nav.cummax() - 1)[base_names].plot(ax=ax)
ax.set_ylabel('Drawdown')
fig.tight_layout()
fig.savefig(str(RESULT / 'drawdown.png'))
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
summary[['gross_total_return', 'net_total_return_approx']].mul(100).plot.bar(ax=ax)
ax.set_ylabel('Total return (%)')
fig.tight_layout()
fig.savefig(str(RESULT / 'gross_net.png'))
plt.show()

fig, ax = plt.subplots(figsize=(12, 5))
quarterly_ic.T.plot.bar(ax=ax)
ax.axhline(0, color='black', linewidth=0.8)
ax.set_ylabel('Mean Rank IC')
fig.tight_layout()
fig.savefig(str(RESULT / 'quarterly_ic.png'))
plt.show()


In [ ]:
group_returns = pd.DataFrame({name: result['groups'].mean() for name, result in results.items()
                              if result['groups'] is not None})
fig, ax = plt.subplots(figsize=(10, 5))
group_returns[['momentum_2', 'ols', 'lightgbm', 'window_ridge', 'lstm_seed42', 'transformer_seed42']].mul(10000).plot.bar(ax=ax)
ax.set_ylabel('Mean period return (bps)')
fig.tight_layout()
fig.savefig(str(RESULT / 'groups.png'))
plt.show()

cost_records = []
for name, result in results.items():
    for bps in [0, 1, 3, 5, 10]:
        daily = result['daily']
        nav = (1 + daily['gross_return'] - daily['traded_ratio'] * bps / 10000).cumprod()
        cost_records.append({'model': name, 'cost_bps': bps, 'net_total_return': nav.iloc[-1] - 1})
cost_table = pd.DataFrame(cost_records).pivot(index='cost_bps', columns='model', values='net_total_return')
fig, ax = plt.subplots(figsize=(12, 5))
cost_table.mul(100).plot(ax=ax, marker='o')
ax.set_ylabel('Net total return (%)')
fig.tight_layout()
fig.savefig(str(RESULT / 'cost_sensitivity.png'))
plt.show()


## 13. 保存本次结果

每个年份写入独立目录：data/ml_alpha_simple/results_年份。
再次运行同一年会覆盖该年的输出。所有预定种子一并保留。


In [ ]:
predictions.to_csv(str(RESULT / 'predictions.csv.gz'), index=False, compression='gzip')
summary.to_csv(str(RESULT / 'summary.csv'))
daily_ic.to_csv(str(RESULT / 'daily_rank_ic.csv'))
training_logs.to_csv(str(RESULT / 'training_log.csv'), index=False)
cost_table.to_csv(str(RESULT / 'cost_sensitivity.csv'))
net_nav.to_csv(str(RESULT / 'net_nav.csv'))
print('结果目录：', RESULT)


## 研究结论需要回答

1. 排序能力是否跨年稳定，是否对随机种子敏感？
2. 复杂模型是否优于简单模型和同信息窗口的线性参照？
3. 成本后是否优于同样本基准，代价是怎样的换手和回撤？
4. 理想成交、事后标签可评价性和多次试验如何限制结论？

只有一个年份的好成绩，不足以证明稳定Alpha。2025已查看，之后的修改都要标为探索。